In [1]:
import os
import pandas as pd
import numpy as np
from PIL import Image, ImageOps
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.multioutput import MultiOutputClassifier
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

BASE_DIR = "/kaggle/input/plant-pathology-2020-fgvc7"
TRAIN_CSV = os.path.join(BASE_DIR, "train.csv")
TEST_CSV = os.path.join(BASE_DIR, "test.csv")
IMAGES_DIR = os.path.join(BASE_DIR, "images")

train_df = pd.read_csv(TRAIN_CSV)
test_df = pd.read_csv(TEST_CSV)


def extract_features(path, size=(300, 300), hist_bins=128):
    """Load an image, resize, compute features for original and flipped version,
    then average them to obtain a more robust representation."""
    img = Image.open(path).convert("RGB")
    img = img.resize(size, Image.BILINEAR)
    img_flip = ImageOps.mirror(img)

    def feat(im):
        arr = np.asarray(im) / 255.0  # (H, W, 3)
        flat = arr.flatten()
        channel_means = arr.mean(axis=(0, 1))
        channel_stds = arr.std(axis=(0, 1))
        hist_features = []
        for c in range(3):
            hist, _ = np.histogram(
                arr[:, :, c], bins=hist_bins, range=(0.0, 1.0), density=True
            )
            hist_features.append(hist)
        hist_features = np.concatenate(hist_features)
        return np.concatenate([flat, channel_means, channel_stds, hist_features])

    f_orig = feat(img)
    f_flip = feat(img_flip)
    return (f_orig + f_flip) / 2.0


train_features = [
    extract_features(os.path.join(IMAGES_DIR, f"{img_id}.jpg"))
    for img_id in train_df["image_id"]
]
X_train_raw = np.stack(train_features)

test_features = [
    extract_features(os.path.join(IMAGES_DIR, f"{img_id}.jpg"))
    for img_id in test_df["image_id"]
]
X_test_raw = np.stack(test_features)

pca = PCA(n_components=0.95, svd_solver="full", random_state=42)
X_train = pca.fit_transform(X_train_raw)
X_test = pca.transform(X_test_raw)

train_labels = train_df[["healthy", "multiple_diseases", "rust", "scab"]].values




In [2]:
# Train full‑data models and create prediction CSVs
rf_base = RandomForestClassifier(
    n_estimators=6000,
    max_depth=None,
    n_jobs=-1,
    class_weight="balanced",
    random_state=42,
)
rf_clf = MultiOutputClassifier(rf_base)
rf_clf.fit(X_train, train_labels)

rf_test_probs = np.stack(
    [estimator.predict_proba(X_test)[:, 1] for estimator in rf_clf.estimators_],
    axis=1,
)

rf_pred_path = "model_predictions_rf.csv"
rf_pred_df = test_df.copy()
rf_pred_df[["healthy", "multiple_diseases", "rust", "scab"]] = rf_test_probs
rf_pred_df.to_csv(rf_pred_path, index=False)
print(f"RandomForest predictions saved to {rf_pred_path}")

logreg_base = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    n_jobs=-1,
    solver="lbfgs",
    C=5.0,
)
logreg_clf = MultiOutputClassifier(logreg_base)
logreg_clf.fit(X_train, train_labels)

logreg_test_probs = np.stack(
    [estimator.predict_proba(X_test)[:, 1] for estimator in logreg_clf.estimators_],
    axis=1,
)

logreg_pred_path = "model_predictions_logreg.csv"
logreg_pred_df = test_df.copy()
logreg_pred_df[["healthy", "multiple_diseases", "rust", "scab"]] = logreg_test_probs
logreg_pred_df.to_csv(logreg_pred_path, index=False)
print(f"LogisticRegression predictions saved to {logreg_pred_path}")




RandomForest predictions saved to model_predictions_rf.csv
LogisticRegression predictions saved to model_predictions_logreg.csv


In [3]:
# Validation split to estimate per‑target performance and compute column‑wise weights
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, train_labels, test_size=0.2, random_state=42
)

rf_val_clf = MultiOutputClassifier(
    RandomForestClassifier(
        n_estimators=6000,
        max_depth=None,
        n_jobs=-1,
        class_weight="balanced",
        random_state=42,
    )
)
rf_val_clf.fit(X_tr, y_tr)
rf_val_probs = np.stack(
    [estimator.predict_proba(X_val)[:, 1] for estimator in rf_val_clf.estimators_],
    axis=1,
)

logreg_val_clf = MultiOutputClassifier(
    LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        n_jobs=-1,
        solver="lbfgs",
        C=5.0,
    )
)
logreg_val_clf.fit(X_tr, y_tr)
logreg_val_probs = np.stack(
    [estimator.predict_proba(X_val)[:, 1] for estimator in logreg_val_clf.estimators_],
    axis=1,
)


def per_class_auc(y_true, probs):
    """Return an array of ROC‑AUC for each target column."""
    return np.array(
        [roc_auc_score(y_true[:, i], probs[:, i]) for i in range(y_true.shape[1])]
    )


rf_aucs = per_class_auc(y_val, rf_val_probs)
logreg_aucs = per_class_auc(y_val, logreg_val_probs)

# Column‑wise ensemble weights: give more weight to the model that scores higher on that column
rf_weights = rf_aucs / (rf_aucs + logreg_aucs)
logreg_weights = 1.0 - rf_weights  # complementary

print(f"Validation AUC per target – RF: {rf_aucs}")
print(f"Validation AUC per target – LogReg: {logreg_aucs}")
print(f"Derived column‑wise weights – RF: {rf_weights}, LogReg: {logreg_weights}")

submissions_all = [rf_pred_path, logreg_pred_path]
weights_per_model = [rf_weights, logreg_weights]  # each is a length‑4 array




Validation AUC per target – RF: [0.72444794 0.58263221 0.61799015 0.61975877]
Validation AUC per target – LogReg: [0.59917966 0.52804487 0.59235468 0.60486842]
Derived column‑wise weights – RF: [0.54732006 0.5245739  0.51059015 0.50607954], LogReg: [0.45267994 0.4754261  0.48940985 0.49392046]


In [4]:
def ensemble(submission_paths, weight_vectors):
    """
    Weighted average of predictions using column‑wise weights.
    `weight_vectors` is a list where each element is an array of length 4
    corresponding to the four target columns.
    """
    if len(submission_paths) != len(weight_vectors):
        raise ValueError("Number of paths and weight vectors must match")
    weighted_sum = None
    for path, w in zip(submission_paths, weight_vectors):
        print(f"Reading {path} with column‑wise weights {w}")
        df = pd.read_csv(path)
        vals = df.loc[:, ["healthy", "multiple_diseases", "rust", "scab"]].values
        # broadcast row‑wise multiplication (rows x 4) * (4,)
        weighted = vals * w
        if weighted_sum is None:
            weighted_sum = weighted
        else:
            weighted_sum += weighted
    return weighted_sum


def make_submission_file(submission_avg, reference_path):
    """
    Write `submission.csv` using the image_id column from a reference file
    and the averaged predictions supplied in `submission_avg`.
    """
    ref_df = pd.read_csv(reference_path)
    if submission_avg.shape[0] != ref_df.shape[0]:
        raise ValueError(
            "Shape mismatch between averaged predictions and reference file"
        )
    submission_df = ref_df.copy()
    submission_df[["healthy", "multiple_diseases", "rust", "scab"]] = submission_avg
    submission_df.to_csv("submission.csv", index=False)
    print("submission.csv written successfully.")


if not submissions_all:
    raise FileNotFoundError("No valid CSV submission files found.")

submission_avg = ensemble(submissions_all, weights_per_model)
make_submission_file(submission_avg, TEST_CSV)

Reading model_predictions_rf.csv with column‑wise weights [0.54732006 0.5245739  0.51059015 0.50607954]
Reading model_predictions_logreg.csv with column‑wise weights [0.45267994 0.4754261  0.48940985 0.49392046]
submission.csv written successfully.
